# Train-Validate-Test Performance Report
**Generated:** 2026-07-09 13:14:20
**Results File:** combined_train_validate_results.csv

This report provides comprehensive analysis of algorithm performance across train, validation, and test sets.

## 📋 Table of Contents

1. [Data Overview](#data-overview)
2. [Train Set Results](#train-set-results)
   - [Best Models by Trait](#train-best-models)
   - [Algorithm Rankings](#train-algorithm-rankings)
   - [Ensemble vs Best Model Comparison](#train-ensemble-comparison)
   - [Performance Tables](#train-performance-tables)
3. [Validation Set Results](#validation-set-results)
   - [Best Models by Trait](#val-best-models)
   - [Algorithm Rankings](#val-algorithm-rankings)
   - [Ensemble vs Best Model Comparison](#val-ensemble-comparison)
   - [Performance Tables](#val-performance-tables)
4. [Test Set Results](#test-set-results)
   - [Best Models by Trait](#test-best-models)
   - [Algorithm Rankings](#test-algorithm-rankings)
   - [Ensemble vs Best Model Comparison](#test-ensemble-comparison)
   - [Performance Tables](#test-performance-tables)
5. [Stacking Preferences](#stacking-preferences)
   - [Per-Trait Preference Profile](#stacking-per-trait)
   - [Stability Analysis](#stacking-stability)
6. [Cross-Set Analysis](#cross-set-analysis)
   - [Performance Across Sets](#performance-across-sets)
   - [Overfitting Analysis](#overfitting-analysis)
7. [Summary & Recommendations](#summary-recommendations)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Set data directory
DATA_DIR = Path('<PROJECT_DIR>/Phase1_Learning_Benchmarking/training_validation')
print(f"📁 Data directory: {DATA_DIR}")

# Load results
results_file = DATA_DIR / 'combined_train_validate_results.csv'
results_df = pd.read_csv(results_file)

# Separate individual algorithms and ensembles
individual_df = results_df[~results_df['algorithm'].str.startswith('Ensemble_')].copy()
ensemble_df = results_df[results_df['algorithm'].str.startswith('Ensemble_')].copy()

print(f'✅ Loaded {len(results_df)} results')
print(f'📊 Traits: {results_df["trait"].nunique()}')
print(f'🔧 Individual Algorithms: {individual_df["algorithm"].nunique()}')
print(f'🎯 Ensemble Methods: {ensemble_df["algorithm"].nunique() if not ensemble_df.empty else 0}')

---

## 📊 Data Overview {#data-overview}

### Dataset Summary

In [ ]:
# Dataset summary
print('=== Dataset Summary ===')
print(f'Total Results: {len(results_df)}')
print(f'Traits Analyzed: {results_df["trait"].nunique()}')
print(f'\nTraits:')
for trait in sorted(results_df['trait'].unique()):
    trait_count = len(results_df[results_df['trait'] == trait])
    print(f'  - {trait}: {trait_count} algorithm results')

print(f'\nAlgorithms Evaluated:')
for alg in sorted(results_df['algorithm'].unique()):
    alg_count = len(results_df[results_df['algorithm'] == alg])
    print(f'  - {alg}: {alg_count} trait results')

# Data split summary
if 'n_train' in results_df.columns:
    sample_row = results_df.iloc[0]
    print(f'\n=== Data Split ===')
    print(f'Train: {int(sample_row["n_train"])} samples ({sample_row["n_train"]/(sample_row["n_train"]+sample_row["n_val"]+sample_row["n_test"])*100:.1f}%)')
    print(f'Validation: {int(sample_row["n_val"])} samples ({sample_row["n_val"]/(sample_row["n_train"]+sample_row["n_val"]+sample_row["n_test"])*100:.1f}%)')
    print(f'Test: {int(sample_row["n_test"])} samples ({sample_row["n_test"]/(sample_row["n_train"]+sample_row["n_val"]+sample_row["n_test"])*100:.1f}%)')

---

## 🎯 Preference (Stacking Weights) {#stacking-preferences}

Convert stacking weights into a publishable preference profile. This section appears when stacking artifacts are present.

### Per-Trait Preference Profile {#stacking-per-trait}

Per trait, report model and family preference as **mean ± SD** across outer folds (and replicates if present), plus a top-k model summary.

In [ ]:
stack_dirs = sorted(DATA_DIR.glob('stacking_*'))
if not stack_dirs:
    print('No stacking artifacts found.')
else:
    fold_frames = []
    model_summary_frames = []
    for d in stack_dirs:
        trait = d.name.replace('stacking_', '')
        fold_file = d / 'stacking_weights_by_fold.csv'
        model_file = d / 'stacking_weights_summary.csv'
        family_file = d / 'stacking_family_weights_summary.csv'
        if fold_file.exists():
            fdf = pd.read_csv(fold_file)
            fdf['trait'] = trait
            if 'replicate' not in fdf.columns:
                fdf['replicate'] = 0
            fold_frames.append(fdf)
        if model_file.exists():
            mdf = pd.read_csv(model_file)
            mdf['trait'] = trait
            model_summary_frames.append(mdf)

    if not fold_frames:
        print('No fold-level stacking files found.')
    else:
        all_folds = pd.concat(fold_frames, ignore_index=True)
        all_folds['weight'] = pd.to_numeric(all_folds['weight'], errors='coerce')
        all_folds = all_folds.dropna(subset=['weight'])

        model_family_map = {}
        if model_summary_frames:
            all_models_summary = pd.concat(model_summary_frames, ignore_index=True)
            for _, row in all_models_summary.iterrows():
                model_family_map[row['model']] = row.get('family', 'other')
        all_folds['family'] = all_folds['model'].map(model_family_map).fillna('other')

        model_pref = all_folds.groupby(['trait', 'model', 'family'], as_index=False)['weight'].agg(['mean', 'std', 'count']).reset_index()
        model_pref = model_pref.rename(columns={'mean': 'mean_weight', 'std': 'sd_weight', 'count': 'n'})
        model_pref['sd_weight'] = model_pref['sd_weight'].fillna(0.0)
        model_pref['mean±SD'] = model_pref['mean_weight'].map(lambda x: f'{x:.4f}') + ' ± ' + model_pref['sd_weight'].map(lambda x: f'{x:.4f}')

        family_pref = all_folds.groupby(['trait', 'family'], as_index=False)['weight'].agg(['mean', 'std', 'count']).reset_index()
        family_pref = family_pref.rename(columns={'mean': 'mean_weight', 'std': 'sd_weight', 'count': 'n'})
        family_pref['sd_weight'] = family_pref['sd_weight'].fillna(0.0)
        family_pref['mean±SD'] = family_pref['mean_weight'].map(lambda x: f'{x:.4f}') + ' ± ' + family_pref['sd_weight'].map(lambda x: f'{x:.4f}')

        top_k = 10
        for trait in sorted(model_pref['trait'].unique()):
            print(f'\n=== {trait}: Model Preference (mean ± SD) ===')
            trait_models = model_pref[model_pref['trait'] == trait].sort_values('mean_weight', ascending=False)
            display(trait_models[['model', 'family', 'mean±SD', 'n']].reset_index(drop=True))

            print(f'\n=== {trait}: Family Preference (mean ± SD) ===')
            trait_families = family_pref[family_pref['trait'] == trait].sort_values('mean_weight', ascending=False)
            display(trait_families[['family', 'mean±SD', 'n']].reset_index(drop=True))

            print(f'\n=== {trait}: Top-{top_k} Models ===')
            display(trait_models.head(top_k)[['model', 'family', 'mean_weight', 'sd_weight']].round(4).reset_index(drop=True))

### Stability Analysis {#stacking-stability}

Summarize stability using coefficient of variation (CV) and visualize fold-to-fold variability.

In [ ]:
stack_dirs = sorted(DATA_DIR.glob('stacking_*'))
if not stack_dirs:
    print('No stacking artifacts found.')
else:
    fold_frames = []
    model_summary_frames = []
    for d in stack_dirs:
        trait = d.name.replace('stacking_', '')
        fold_file = d / 'stacking_weights_by_fold.csv'
        model_file = d / 'stacking_weights_summary.csv'
        if fold_file.exists():
            fdf = pd.read_csv(fold_file)
            fdf['trait'] = trait
            if 'replicate' not in fdf.columns:
                fdf['replicate'] = 0
            fold_frames.append(fdf)
        if model_file.exists():
            mdf = pd.read_csv(model_file)
            mdf['trait'] = trait
            model_summary_frames.append(mdf)

    if not fold_frames:
        print('No fold-level stacking files found.')
    else:
        all_folds = pd.concat(fold_frames, ignore_index=True)
        all_folds['weight'] = pd.to_numeric(all_folds['weight'], errors='coerce')
        all_folds = all_folds.dropna(subset=['weight'])

        model_family_map = {}
        if model_summary_frames:
            all_models_summary = pd.concat(model_summary_frames, ignore_index=True)
            for _, row in all_models_summary.iterrows():
                model_family_map[row['model']] = row.get('family', 'other')
        all_folds['family'] = all_folds['model'].map(model_family_map).fillna('other')

        stability = all_folds.groupby(['trait', 'model', 'family'], as_index=False)['weight'].agg(['mean', 'std']).reset_index()
        stability = stability.rename(columns={'mean': 'mean_weight', 'std': 'sd_weight'})
        stability['sd_weight'] = stability['sd_weight'].fillna(0.0)
        eps = 1e-12
        stability['cv'] = stability['sd_weight'] / np.maximum(np.abs(stability['mean_weight']), eps)

        print('=== Stability Metrics (Coefficient of Variation) ===')
        display(stability.sort_values(['trait', 'cv'], ascending=[True, False]).round(4))

        for trait in sorted(all_folds['trait'].unique()):
            trait_df = all_folds[all_folds['trait'] == trait].copy()
            trait_mean = trait_df.groupby('model', as_index=False)['weight'].mean().sort_values('weight', ascending=False)
            top_models = trait_mean.head(10)['model'].tolist()
            plot_df = trait_df[trait_df['model'].isin(top_models)].copy()
            if plot_df.empty:
                continue
            plot_df = plot_df.groupby(['replicate', 'fold', 'model'], as_index=False)['weight'].mean()

            plt.figure(figsize=(12, 5))
            sns.lineplot(data=plot_df, x='fold', y='weight', hue='model', marker='o')
            plt.title(f'{trait}: Fold-to-Fold Variability (Top 10 Models)')
            plt.xlabel('Outer Fold')
            plt.ylabel('Weight')
            plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()

            heat_df = plot_df.pivot_table(index='model', columns='fold', values='weight', aggfunc='mean')
            plt.figure(figsize=(10, max(4, 0.4 * len(heat_df))))
            sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='viridis')
            plt.title(f'{trait}: Outer-Fold Weight Heatmap (Top 10 Models)')
            plt.xlabel('Outer Fold')
            plt.ylabel('Model')
            plt.tight_layout()
            plt.show()

---

## Train Results {#train-set}

This section presents all results for the **train** set.

### Best Models by Trait {#train-best-models}

Best performing algorithm for each trait on the train set (ranked by Train R²).

In [ ]:
# Best model per trait on train set
best_col = 'train_r2'
best_per_trait = results_df.loc[results_df.groupby('trait')[best_col].idxmax()].copy()
best_per_trait = best_per_trait.sort_values(best_col, ascending=False)

# Create comprehensive table
best_table = best_per_trait[[
    'trait', 'algorithm',
    'train_r2', 'train_pearson_r',
    'train_rmse', 'train_mae', 'train_bias'
]].round(4)

# Rename columns for display
best_table.columns = ['Trait', 'Best Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print('=== Best Algorithm per Trait (Train Set) ===')
print('Ranked by Train R²')
print()
display(HTML(best_table.to_html(index=False, classes='table table-striped')))

# Summary statistics
print(f'\n=== Summary Statistics ===')
print(f'Mean R²: 1.0000')
print(f'Median R²: 1.0000')
print(f'Best R²: 1.0000')
print(f'Worst R²: 0.9999')

### Algorithm Rankings {#train-algorithm-rankings}

Average performance across all traits for each algorithm on the train set.

In [ ]:
# Algorithm rankings on train set
r2_col = 'train_r2'
pearson_col = 'train_pearson_r'
rmse_col = 'train_rmse'
mae_col = 'train_mae'
alg_rankings = individual_df.groupby('algorithm').agg({
    r2_col: ['mean', 'std', 'min', 'max'],
    pearson_col: ['mean', 'std'],
    rmse_col: 'mean',
    mae_col: 'mean'
}).round(4)

# Flatten column names
alg_rankings.columns = ['_'.join(col).strip() for col in alg_rankings.columns.values]
r2_mean_col = f'{r2_col}_mean'
r2_std_col = f'{r2_col}_std'
alg_rankings = alg_rankings.sort_values(r2_mean_col, ascending=False)

# Rename for display
pearson_mean_col = f'{pearson_col}_mean'
rmse_mean_col = f'{rmse_col}_mean'
mae_mean_col = f'{mae_col}_mean'
display_cols = [r2_mean_col, r2_std_col, pearson_mean_col, rmse_mean_col, mae_mean_col]
display_table = alg_rankings[display_cols].copy()
display_table.columns = ['R² (Mean)', 'R² (Std)', 'Pearson r (Mean)', 'RMSE (Mean)', 'MAE (Mean)']

print('=== Algorithm Rankings (Train Set) ===')
display(HTML(display_table.to_html(classes='table table-striped')))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(alg_rankings))
ax.barh(x, alg_rankings[r2_mean_col], 
        xerr=alg_rankings[r2_std_col], capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(alg_rankings.index, rotation=0)
ax.set_xlabel('R² (Mean ± Std)')
ax.set_title(f'Algorithm Performance Ranking (Train Set)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Ensemble vs Best Model Comparison {#train-ensemble-comparison}

Comparison of ensemble methods against the best individual algorithm for each trait on the train set.

In [ ]:
# Ensemble vs Best Model Comparison on train set
if not ensemble_df.empty:
    comparison_data = []
    
    for trait in sorted(results_df['trait'].unique()):
        trait_individual = individual_df[individual_df['trait'] == trait]
        trait_ensemble = ensemble_df[ensemble_df['trait'] == trait]
        
        if len(trait_individual) > 0 and len(trait_ensemble) > 0:
            # Best individual algorithm
            r2_col = 'train_r2'
            best_individual = trait_individual.loc[trait_individual[r2_col].idxmax()]
            
            # Best ensemble
            best_ensemble = trait_ensemble.loc[trait_ensemble[r2_col].idxmax()]
            
            # Calculate improvement
            individual_r2 = best_individual[r2_col]
            ensemble_r2 = best_ensemble[r2_col]
            improvement = ensemble_r2 - individual_r2
            improvement_pct = (improvement / abs(individual_r2) * 100) if individual_r2 != 0 else 0
            
            comparison_data.append({
                'Trait': trait,
                'Best Individual': best_individual['algorithm'],
                'Best Individual R²': individual_r2,
                'Best Ensemble': best_ensemble['algorithm'],
                'Best Ensemble R²': ensemble_r2,
                'Improvement': improvement,
                'Improvement %': improvement_pct
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Improvement', ascending=False)
    comparison_df = comparison_df.round(4)
    
    print('=== Ensemble vs Best Individual Algorithm (Train Set) ===')
    display(HTML(comparison_df.to_html(index=False, classes='table table-striped')))
    
    # Summary
    print('\n=== Summary ===')
    better_count = (comparison_df['Improvement'] > 0).sum()
    worse_count = (comparison_df['Improvement'] < 0).sum()
    mean_improvement = comparison_df['Improvement'].mean()
    median_improvement = comparison_df['Improvement'].median()
    print(f'Traits where ensemble is better: {better_count}')
    print(f'Traits where individual is better: {worse_count}')
    print(f'Mean improvement: {mean_improvement:.4f}')
    print(f'Median improvement: {median_improvement:.4f}')
    
    # Visualization
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(comparison_df))
    width = 0.35
    ax.bar(x - width/2, comparison_df['Best Individual R²'], width, label='Best Individual', alpha=0.8)
    ax.bar(x + width/2, comparison_df['Best Ensemble R²'], width, label='Best Ensemble', alpha=0.8)
    ax.set_xlabel('Trait')
    ax.set_ylabel('R²')
    ax.set_title(f'Ensemble vs Best Individual Algorithm (Train Set)')
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df['Trait'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No ensemble results available')

### Complete Performance Tables {#train-performance-tables}

Detailed performance metrics for all algorithms and traits on the train set.

In [ ]:
# Complete performance table for train set
perf_cols = ['trait', 'algorithm',
            f'train_r2', f'train_pearson_r',
            f'train_rmse', f'train_mae', f'train_bias']

perf_table = results_df[perf_cols].copy()
perf_table = perf_table.sort_values(['trait', f'train_r2'], ascending=[True, False])
perf_table = perf_table.round(4)

# Rename columns
perf_table.columns = ['Trait', 'Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print(f'=== Complete Performance Table (Train Set) ===')
print(f'Total entries: {len(perf_table)}')
print()

# Display by trait for better readability
for trait_name in sorted(perf_table['Trait'].unique()):
    trait_data = perf_table[perf_table['Trait'] == trait_name].copy()
    print(f'\n**{trait_name}**')
    display(HTML(trait_data.to_html(index=False, classes='table table-striped table-sm')))

---

## Validation Results {#validation-set}

This section presents all results for the **validation** set.

### Best Models by Trait {#validation-best-models}

Best performing algorithm for each trait on the validation set (ranked by Validation R²).

In [ ]:
# Best model per trait on validation set
best_col = 'val_r2'
best_per_trait = results_df.loc[results_df.groupby('trait')[best_col].idxmax()].copy()
best_per_trait = best_per_trait.sort_values(best_col, ascending=False)

# Create comprehensive table
best_table = best_per_trait[[
    'trait', 'algorithm',
    'val_r2', 'val_pearson_r',
    'val_rmse', 'val_mae', 'val_bias'
]].round(4)

# Rename columns for display
best_table.columns = ['Trait', 'Best Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print('=== Best Algorithm per Trait (Validation Set) ===')
print('Ranked by Validation R²')
print()
display(HTML(best_table.to_html(index=False, classes='table table-striped')))

# Summary statistics
print(f'\n=== Summary Statistics ===')
print(f'Mean R²: 0.7819')
print(f'Median R²: 0.8099')
print(f'Best R²: 0.8177')
print(f'Worst R²: 0.6901')

### Algorithm Rankings {#validation-algorithm-rankings}

Average performance across all traits for each algorithm on the validation set.

In [ ]:
# Algorithm rankings on validation set
r2_col = 'val_r2'
pearson_col = 'val_pearson_r'
rmse_col = 'val_rmse'
mae_col = 'val_mae'
alg_rankings = individual_df.groupby('algorithm').agg({
    r2_col: ['mean', 'std', 'min', 'max'],
    pearson_col: ['mean', 'std'],
    rmse_col: 'mean',
    mae_col: 'mean'
}).round(4)

# Flatten column names
alg_rankings.columns = ['_'.join(col).strip() for col in alg_rankings.columns.values]
r2_mean_col = f'{r2_col}_mean'
r2_std_col = f'{r2_col}_std'
alg_rankings = alg_rankings.sort_values(r2_mean_col, ascending=False)

# Rename for display
pearson_mean_col = f'{pearson_col}_mean'
rmse_mean_col = f'{rmse_col}_mean'
mae_mean_col = f'{mae_col}_mean'
display_cols = [r2_mean_col, r2_std_col, pearson_mean_col, rmse_mean_col, mae_mean_col]
display_table = alg_rankings[display_cols].copy()
display_table.columns = ['R² (Mean)', 'R² (Std)', 'Pearson r (Mean)', 'RMSE (Mean)', 'MAE (Mean)']

print('=== Algorithm Rankings (Validation Set) ===')
display(HTML(display_table.to_html(classes='table table-striped')))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(alg_rankings))
ax.barh(x, alg_rankings[r2_mean_col], 
        xerr=alg_rankings[r2_std_col], capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(alg_rankings.index, rotation=0)
ax.set_xlabel('R² (Mean ± Std)')
ax.set_title(f'Algorithm Performance Ranking (Validation Set)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Ensemble vs Best Model Comparison {#validation-ensemble-comparison}

Comparison of ensemble methods against the best individual algorithm for each trait on the validation set.

In [ ]:
# Ensemble vs Best Model Comparison on validation set
if not ensemble_df.empty:
    comparison_data = []
    
    for trait in sorted(results_df['trait'].unique()):
        trait_individual = individual_df[individual_df['trait'] == trait]
        trait_ensemble = ensemble_df[ensemble_df['trait'] == trait]
        
        if len(trait_individual) > 0 and len(trait_ensemble) > 0:
            # Best individual algorithm
            r2_col = 'val_r2'
            best_individual = trait_individual.loc[trait_individual[r2_col].idxmax()]
            
            # Best ensemble
            best_ensemble = trait_ensemble.loc[trait_ensemble[r2_col].idxmax()]
            
            # Calculate improvement
            individual_r2 = best_individual[r2_col]
            ensemble_r2 = best_ensemble[r2_col]
            improvement = ensemble_r2 - individual_r2
            improvement_pct = (improvement / abs(individual_r2) * 100) if individual_r2 != 0 else 0
            
            comparison_data.append({
                'Trait': trait,
                'Best Individual': best_individual['algorithm'],
                'Best Individual R²': individual_r2,
                'Best Ensemble': best_ensemble['algorithm'],
                'Best Ensemble R²': ensemble_r2,
                'Improvement': improvement,
                'Improvement %': improvement_pct
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Improvement', ascending=False)
    comparison_df = comparison_df.round(4)
    
    print('=== Ensemble vs Best Individual Algorithm (Validation Set) ===')
    display(HTML(comparison_df.to_html(index=False, classes='table table-striped')))
    
    # Summary
    print('\n=== Summary ===')
    better_count = (comparison_df['Improvement'] > 0).sum()
    worse_count = (comparison_df['Improvement'] < 0).sum()
    mean_improvement = comparison_df['Improvement'].mean()
    median_improvement = comparison_df['Improvement'].median()
    print(f'Traits where ensemble is better: {better_count}')
    print(f'Traits where individual is better: {worse_count}')
    print(f'Mean improvement: {mean_improvement:.4f}')
    print(f'Median improvement: {median_improvement:.4f}')
    
    # Visualization
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(comparison_df))
    width = 0.35
    ax.bar(x - width/2, comparison_df['Best Individual R²'], width, label='Best Individual', alpha=0.8)
    ax.bar(x + width/2, comparison_df['Best Ensemble R²'], width, label='Best Ensemble', alpha=0.8)
    ax.set_xlabel('Trait')
    ax.set_ylabel('R²')
    ax.set_title(f'Ensemble vs Best Individual Algorithm (Validation Set)')
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df['Trait'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No ensemble results available')

### Complete Performance Tables {#validation-performance-tables}

Detailed performance metrics for all algorithms and traits on the validation set.

In [ ]:
# Complete performance table for validation set
perf_cols = ['trait', 'algorithm',
            f'val_r2', f'val_pearson_r',
            f'val_rmse', f'val_mae', f'val_bias']

perf_table = results_df[perf_cols].copy()
perf_table = perf_table.sort_values(['trait', f'val_r2'], ascending=[True, False])
perf_table = perf_table.round(4)

# Rename columns
perf_table.columns = ['Trait', 'Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print(f'=== Complete Performance Table (Validation Set) ===')
print(f'Total entries: {len(perf_table)}')
print()

# Display by trait for better readability
for trait_name in sorted(perf_table['Trait'].unique()):
    trait_data = perf_table[perf_table['Trait'] == trait_name].copy()
    print(f'\n**{trait_name}**')
    display(HTML(trait_data.to_html(index=False, classes='table table-striped table-sm')))

---

## Test Results {#test-set}

This section presents all results for the **test** set.

### Best Models by Trait {#test-best-models}

Best performing algorithm for each trait on the test set (ranked by Test R²).

In [ ]:
# Best model per trait on test set
best_col = 'test_r2'
best_per_trait = results_df.loc[results_df.groupby('trait')[best_col].idxmax()].copy()
best_per_trait = best_per_trait.sort_values(best_col, ascending=False)

# Create comprehensive table
best_table = best_per_trait[[
    'trait', 'algorithm',
    'test_r2', 'test_pearson_r',
    'test_rmse', 'test_mae', 'test_bias'
]].round(4)

# Rename columns for display
best_table.columns = ['Trait', 'Best Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print('=== Best Algorithm per Trait (Test Set) ===')
print('Ranked by Test R²')
print()
display(HTML(best_table.to_html(index=False, classes='table table-striped')))

# Summary statistics
print(f'\n=== Summary Statistics ===')
print(f'Mean R²: 0.7829')
print(f'Median R²: 0.8066')
print(f'Best R²: 0.8303')
print(f'Worst R²: 0.6882')

### Algorithm Rankings {#test-algorithm-rankings}

Average performance across all traits for each algorithm on the test set.

In [ ]:
# Algorithm rankings on test set
r2_col = 'test_r2'
pearson_col = 'test_pearson_r'
rmse_col = 'test_rmse'
mae_col = 'test_mae'
alg_rankings = individual_df.groupby('algorithm').agg({
    r2_col: ['mean', 'std', 'min', 'max'],
    pearson_col: ['mean', 'std'],
    rmse_col: 'mean',
    mae_col: 'mean'
}).round(4)

# Flatten column names
alg_rankings.columns = ['_'.join(col).strip() for col in alg_rankings.columns.values]
r2_mean_col = f'{r2_col}_mean'
r2_std_col = f'{r2_col}_std'
alg_rankings = alg_rankings.sort_values(r2_mean_col, ascending=False)

# Rename for display
pearson_mean_col = f'{pearson_col}_mean'
rmse_mean_col = f'{rmse_col}_mean'
mae_mean_col = f'{mae_col}_mean'
display_cols = [r2_mean_col, r2_std_col, pearson_mean_col, rmse_mean_col, mae_mean_col]
display_table = alg_rankings[display_cols].copy()
display_table.columns = ['R² (Mean)', 'R² (Std)', 'Pearson r (Mean)', 'RMSE (Mean)', 'MAE (Mean)']

print('=== Algorithm Rankings (Test Set) ===')
display(HTML(display_table.to_html(classes='table table-striped')))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(alg_rankings))
ax.barh(x, alg_rankings[r2_mean_col], 
        xerr=alg_rankings[r2_std_col], capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(alg_rankings.index, rotation=0)
ax.set_xlabel('R² (Mean ± Std)')
ax.set_title(f'Algorithm Performance Ranking (Test Set)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Ensemble vs Best Model Comparison {#test-ensemble-comparison}

Comparison of ensemble methods against the best individual algorithm for each trait on the test set.

In [ ]:
# Ensemble vs Best Model Comparison on test set
if not ensemble_df.empty:
    comparison_data = []
    
    for trait in sorted(results_df['trait'].unique()):
        trait_individual = individual_df[individual_df['trait'] == trait]
        trait_ensemble = ensemble_df[ensemble_df['trait'] == trait]
        
        if len(trait_individual) > 0 and len(trait_ensemble) > 0:
            # Best individual algorithm
            r2_col = 'test_r2'
            best_individual = trait_individual.loc[trait_individual[r2_col].idxmax()]
            
            # Best ensemble
            best_ensemble = trait_ensemble.loc[trait_ensemble[r2_col].idxmax()]
            
            # Calculate improvement
            individual_r2 = best_individual[r2_col]
            ensemble_r2 = best_ensemble[r2_col]
            improvement = ensemble_r2 - individual_r2
            improvement_pct = (improvement / abs(individual_r2) * 100) if individual_r2 != 0 else 0
            
            comparison_data.append({
                'Trait': trait,
                'Best Individual': best_individual['algorithm'],
                'Best Individual R²': individual_r2,
                'Best Ensemble': best_ensemble['algorithm'],
                'Best Ensemble R²': ensemble_r2,
                'Improvement': improvement,
                'Improvement %': improvement_pct
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Improvement', ascending=False)
    comparison_df = comparison_df.round(4)
    
    print('=== Ensemble vs Best Individual Algorithm (Test Set) ===')
    display(HTML(comparison_df.to_html(index=False, classes='table table-striped')))
    
    # Summary
    print('\n=== Summary ===')
    better_count = (comparison_df['Improvement'] > 0).sum()
    worse_count = (comparison_df['Improvement'] < 0).sum()
    mean_improvement = comparison_df['Improvement'].mean()
    median_improvement = comparison_df['Improvement'].median()
    print(f'Traits where ensemble is better: {better_count}')
    print(f'Traits where individual is better: {worse_count}')
    print(f'Mean improvement: {mean_improvement:.4f}')
    print(f'Median improvement: {median_improvement:.4f}')
    
    # Visualization
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(comparison_df))
    width = 0.35
    ax.bar(x - width/2, comparison_df['Best Individual R²'], width, label='Best Individual', alpha=0.8)
    ax.bar(x + width/2, comparison_df['Best Ensemble R²'], width, label='Best Ensemble', alpha=0.8)
    ax.set_xlabel('Trait')
    ax.set_ylabel('R²')
    ax.set_title(f'Ensemble vs Best Individual Algorithm (Test Set)')
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df['Trait'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No ensemble results available')

### Complete Performance Tables {#test-performance-tables}

Detailed performance metrics for all algorithms and traits on the test set.

In [ ]:
# Complete performance table for test set
perf_cols = ['trait', 'algorithm',
            f'test_r2', f'test_pearson_r',
            f'test_rmse', f'test_mae', f'test_bias']

perf_table = results_df[perf_cols].copy()
perf_table = perf_table.sort_values(['trait', f'test_r2'], ascending=[True, False])
perf_table = perf_table.round(4)

# Rename columns
perf_table.columns = ['Trait', 'Algorithm', 'R²', 'Pearson r', 'RMSE', 'MAE', 'Bias']

print(f'=== Complete Performance Table (Test Set) ===')
print(f'Total entries: {len(perf_table)}')
print()

# Display by trait for better readability
for trait_name in sorted(perf_table['Trait'].unique()):
    trait_data = perf_table[perf_table['Trait'] == trait_name].copy()
    print(f'\n**{trait_name}**')
    display(HTML(trait_data.to_html(index=False, classes='table table-striped table-sm')))

---

## 📈 Cross-Set Analysis {#cross-set-analysis}

### Performance Across Sets

In [ ]:
# Performance across sets
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# R² comparison
individual_df = results_df[~results_df['algorithm'].str.startswith('Ensemble_')]

for idx, (set_name, col) in enumerate([('Train', 'train_r2'), ('Validation', 'val_r2'), ('Test', 'test_r2')]):
    data = [individual_df[individual_df['trait'] == trait][col].values 
            for trait in sorted(individual_df['trait'].unique())]
    axes[idx].boxplot(data, labels=sorted(individual_df['trait'].unique()))
    axes[idx].set_title(f'{set_name} R² by Trait')
    axes[idx].set_ylabel('R²')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Algorithm performance across sets
alg_cross = individual_df.groupby('algorithm').agg({
    'train_r2': 'mean',
    'val_r2': 'mean',
    'test_r2': 'mean'
}).round(4).sort_values('test_r2', ascending=False)

print('=== Algorithm Performance Across Sets (Mean R²) ===')
display(HTML(alg_cross.to_html(classes='table table-striped')))

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(alg_cross))
width = 0.25
ax.bar(x - width, alg_cross['train_r2'], width, label='Train R²', alpha=0.8)
ax.bar(x, alg_cross['val_r2'], width, label='Validation R²', alpha=0.8)
ax.bar(x + width, alg_cross['test_r2'], width, label='Test R²', alpha=0.8)
ax.set_xlabel('Algorithm')
ax.set_ylabel('R²')
ax.set_title('Algorithm Performance Comparison Across Sets')
ax.set_xticks(x)
ax.set_xticklabels(alg_cross.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Overfitting Analysis {#overfitting-analysis}

Overfitting indicator = Train R² - Test R²

In [ ]:
# Overfitting analysis
results_df['overfitting'] = results_df['train_r2'] - results_df['test_r2']

overfit_by_alg = individual_df.groupby('algorithm')['overfitting_indicator'].agg(['mean', 'std']).round(4)
overfit_by_alg = overfit_by_alg.sort_values('mean')

print('=== Overfitting Analysis by Algorithm ===')
display(HTML(overfit_by_alg.to_html(classes='table table-striped')))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(overfit_by_alg))
ax.barh(x, overfit_by_alg['mean'], xerr=overfit_by_alg['std'], capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(overfit_by_alg.index)
ax.set_xlabel('Overfitting Indicator (Train R² - Test R²)')
ax.set_title('Overfitting Analysis by Algorithm')
ax.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='No Overfitting')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---

## 📊 Summary & Recommendations {#summary-recommendations}

### Overall Best Performers

In [ ]:
# Overall summary
print('=== Overall Summary ===')
print()

# Best algorithm overall (by test R²)
best_overall = individual_df.loc[individual_df['test_r2'].idxmax()]
print(f'Best Overall Algorithm (Test Set):')
print(f'  Algorithm: {best_overall["algorithm"]}')
print(f'  Trait: {best_overall["trait"]}')
print(f'  Test R²: {best_overall["test_r2"]:.4f}')
print(f'  Test Pearson r: {best_overall["test_pearson_r"]:.4f}')
print()

# Best algorithm on average
best_avg = individual_df.groupby('algorithm')['test_r2'].mean().sort_values(ascending=False)
print(f'Best Algorithm on Average (across all traits):')
print(f'  Algorithm: {best_avg.index[0]}')
print(f'  Mean Test R²: {best_avg.iloc[0]:.4f}')
print()

# Ensemble summary
if not ensemble_df.empty:
    best_ensemble_avg = ensemble_df.groupby('algorithm')['test_r2'].mean().sort_values(ascending=False)
    print(f'Best Ensemble on Average:')
    print(f'  Ensemble: {best_ensemble_avg.index[0]}')
    print(f'  Mean Test R²: {best_ensemble_avg.iloc[0]:.4f}')
    print()
    
    # Compare ensemble to best individual
    ensemble_improvement = best_ensemble_avg.iloc[0] - best_avg.iloc[0]
    print(f'Ensemble vs Best Individual:')
    print(f'  Improvement: {ensemble_improvement:.4f}')
    if ensemble_improvement > 0:
        print(f'  ✅ Ensemble performs better on average')
    else:
        print(f'  ⚠️  Individual algorithm performs better on average')

print()
print('=== Recommendations ===')
print('1. Review the detailed tables above to find the best algorithm for each trait')
print('2. Consider ensemble methods if they show consistent improvement')
print('3. Check overfitting indicators to ensure model generalization')
print('4. Use validation set performance for model selection')
print('5. Test set performance provides final unbiased evaluation')